In [9]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from sklearn.manifold import MDS
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities, modularity


input_dir = 'data/'

In [10]:
symbols = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'DOGEUSDT']

In [11]:
candlestick_cols = [
    "Open time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close time",
    "Quote asset volume",
    "Number of trades",
    "Taker buy base asset volume",
    "Taker buy quote asset volume",
    "Ignore"
]

In [12]:
def get_sorted_cols(cols):
    # Convert names → indices
    pairs = [(name, candlestick_cols.index(name)) for name in cols]

    # Sort by the index
    pairs_sorted = sorted(pairs, key=lambda x: x[1])

    # Extract sorted names and indices
    sorted_names  = [name for name, idx in pairs_sorted]
    sorted_indices = [idx for name, idx in pairs_sorted]

    return sorted_names, sorted_indices

In [13]:
def load_series(symbol, cols):
    time_serieses = []

    sorted_names, sorted_indices = get_sorted_cols(cols)
    col_names = [f'{symbol}_{name}' for name in sorted_names]
    df = pd.read_csv(
            os.path.join(input_dir, f"{symbol}.csv"),
            usecols=[0, *sorted_indices],
            header = None,
            names=["time", *col_names]
        )
    df.loc[df["time"] >= 1e15, "time"] = df["time"] // 1000
    df["time"] = pd.to_datetime(df["time"], unit="ms")
    df = df[df["time"] >= pd.Timestamp('2023-03-24 14:00:00')]
    for name in col_names:
        s = pd.Series(
            df[name].values,
            index = df["time"],
            name = name
        )
        time_serieses.append(s)
    return time_serieses

### IMPORTANT ###
### IMPORTANT ###
### IMPORTANT ###

load_dataframes function takes in n symbols and the m column indices, and returns a tuple of m dataframes, each with n columns corresponding to the requested symbols. For example, load_dataframes(['BTCUSDT', 'ETHUSDT'], ['Open', "Quote asset volume"]) will return two dataframes: the first containing the 'Open' prices for BTCUSDT, ETHUSDT and the second containing the 'Quote asset volume' for the same symbols. Don't pass in "Open time" as this will automatically as the index

In [14]:
def load_dataframes(symbol_subset, cols):
    dfs = [[] for _ in cols]
    for symbol in symbol_subset:
        time_serieses = load_series(symbol, cols)
        for i, time_series in enumerate(time_serieses):
            dfs[i].append(time_series)
    for i, col_index in enumerate(cols):
        dfs[i] = pd.concat(dfs[i], axis=1)
    return dfs

price, = load_dataframes(['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'DOGEUSDT'], ['Open'])

In [17]:
price.to_csv("timeseries.csv")